In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train.shape, test.shape

((1460, 81), (1459, 80))

In [22]:
# Defensive: drop these if they exist
cols_to_drop = [c for c in ["SalePrice", "Id"] if c in train.columns]
X = train.drop(cols_to_drop, axis=1)
y = train["SalePrice"]

# Sanity check
assert "Id" not in X.columns, "Id still in X!"
assert "SalePrice" not in X.columns, "SalePrice still in X!"
print(f"X shape: {X.shape} (expected 79 cols)")

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X shape: (1460, 79) (expected 79 cols)


In [23]:
# Define hyperparameter search space:
param_distributions = {
    "regressor__n_estimators": [100, 200, 500, 1000],
    "regressor__max_depth":    [None, 10, 20, 30, 50],
    "regressor__max_features": ["sqrt", "log2", 1.0],
    "regressor__min_samples_split": [2,5,10],
    "regressor__min_samples_leaf": [1,2,4]
}

In [24]:
# Run RandomizedSearchCV

import time
start = time.time()

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(X_train, y_train)

print(f"\n[DONE] Took {time.time() - start:.1f} seconds")
print(f"Best CV MAE: ${-search.best_score_:,.0f}")
print(f"Best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits

[DONE] Took 59.2 seconds
Best CV MAE: $17,914
Best params:
  regressor__n_estimators: 500
  regressor__min_samples_split: 2
  regressor__min_samples_leaf: 1
  regressor__max_features: sqrt
  regressor__max_depth: None


In [25]:
# RandomizedSearchCV automatically refits the best model on all training data
best_model = search.best_estimator_

# Score on the held-out validation set
y_pred = best_model.predict(X_val)
val_mae = mean_absolute_error(y_val, y_pred)
val_r2 = r2_score(y_val, y_pred)

print(f"Tuned model validation MAE: ${val_mae:,.0f}")
print(f"Tuned model validation R²:  {val_r2:.2f}")

Tuned model validation MAE: $18,491
Tuned model validation R²:  0.86


In [26]:
# Load the real test data (we already imported it in Cell 1, but reloading for clarity)
test = pd.read_csv("../data/test.csv")

# Keep the Id column — Kaggle needs it for submission
test_ids = test["Id"]

# Apply the same drops we did to X (drop Id, keep all other columns)
X_test = test.drop("Id", axis=1)

print(f"X_test shape: {X_test.shape}")
print(f"X shape (train features): {X.shape}")

X_test shape: (1459, 79)
X shape (train features): (1460, 79)


In [30]:
# Use the best tuned model to predict on the real test set
test_predictions = best_model.predict(X_test)

print(f"Generated {len(test_predictions)} predictions")
print(f"Sample predictions: {test_predictions[:5]}")
print(f"Min prediction:  ${test_predictions.min():,.0f}")
print(f"Max prediction:  ${test_predictions.max():,.0f}")
print(f"Mean prediction: ${test_predictions.mean():,.0f}")

Generated 1459 predictions
Sample predictions: [127841.136 153508.92  183987.7   194649.276 202539.722]
Min prediction:  $73,829
Max prediction:  $432,277
Mean prediction: $179,259


In [31]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_predictions
})

submission.to_csv("../submission.csv", index=False)
print("Submission saved!")
submission.head()

Submission saved!


,Id,SalePrice
0,1461,127841.136
1,1462,153508.920
2,1463,183987.700
3,1464,194649.276
4,1465,202539.722


In [32]:
check = pd.read_csv("../submission.csv")
print(f"Shape: {check.shape}")
print(f"Columns: {list(check.columns)}")
print(f"Any NaN predictions? {check['SalePrice'].isna().any()}")
check.head()

Shape: (1459, 2)
Columns: ['Id', 'SalePrice']
Any NaN predictions? False


,Id,SalePrice
0,1461,127841.136
1,1462,153508.920
2,1463,183987.700
3,1464,194649.276
4,1465,202539.722


[CV] END regressor__max_depth=None, regressor__max_features=1.0, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=200; total time=   4.2s
[CV] END regressor__max_depth=50, regressor__max_features=log2, regressor__min_samples_leaf=1, regressor__min_samples_split=2, regressor__n_estimators=200; total time=   0.8s
[CV] END regressor__max_depth=None, regressor__max_features=1.0, regressor__min_samples_leaf=1, regressor__min_samples_split=5, regressor__n_estimators=200; total time=   3.3s
[CV] END regressor__max_depth=20, regressor__max_features=1.0, regressor__min_samples_leaf=1, regressor__min_samples_split=10, regressor__n_estimators=200; total time=   2.4s
[CV] END regressor__max_depth=30, regressor__max_features=sqrt, regressor__min_samples_leaf=2, regressor__min_samples_split=5, regressor__n_estimators=500; total time=   1.2s
[CV] END regressor__max_depth=30, regressor__max_features=log2, regressor__min_samples_leaf=2, regressor__min_samples_split